# 🚀 AI-901 Hands-on Lab: Azure AI & Model Alternatif

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammad-Ikhwan-Fathulloh/AI-901-Foundry-Labs/blob/main/AI_901_Hands_On_Lab.ipynb)

Notebook ini memungkinkan Anda mencoba seluruh modul praktik kurikulum **AI-901** secara interaktif di browser melalui Google Colab:
- **Modul 00:** LLM 100% Gratis Tanpa API Key (Hugging Face Transformers)
- **Modul 01:** Chat Client Interaktif (Multi-turn Conversation)
- **Modul 02:** Autonomous AI Agent dengan Tool/Function Calling
- **Modul 03:** Analisis Teks Berstruktur (JSON Mode & NER)
- **Modul 04:** Layanan Suara Gratis (Edge-TTS & Audio Player)
- **Modul 05:** Multimodal Computer Vision (Analisis Foto Rak Toko)
- **Modul 06:** Document Understanding (Ekstraksi Faktur Invoice PDF)

## 🛠️ Langkah 1: Kloning Repositori & Instalasi Dependensi

In [ ]:
# Kloning repositori resmi AI-901
!git clone https://github.com/Muhammad-Ikhwan-Fathulloh/AI-901-Foundry-Labs.git
%cd AI-901-Foundry-Labs

# Pasang dependensi yang dibutuhkan
!pip install -q openai python-dotenv edge-tts pypdf requests transformers torch
print("✅ Setup lingkungan Colab selesai!")

## 🔑 Langkah 2: Konfigurasi API Key (Pilih Salah Satu)

> **💡 Rekomendasi:** Jika tidak memiliki langganan Azure, gunakan **Google Gemini (0 Rupiah / Free Tier)** dari [Google AI Studio](https://aistudio.google.com/app/apikey).
> Anda juga dapat memasukkan kunci melalui fitur **Colab Secrets** (ikon kunci 🔑 di sidebar kiri).

In [ ]:
import os

# Coba baca dari Colab Secrets jika ada, jika tidak minta input interaktif
try:
    from google.colab import userdata
    gemini_key = userdata.get('GEMINI_API_KEY')
except Exception:
    gemini_key = None

if not gemini_key:
    # Masukkan kunci Gemini Anda di sini atau biarkan kosong jika hanya ingin coba Modul 00 (Tanpa Key)
    gemini_key = input("Masukkan GEMINI_API_KEY (tekan Enter jika ingin lewati): ").strip()

if gemini_key:
    os.environ["GEMINI_API_KEY"] = gemini_key
    os.environ["AI_PROVIDER"] = "gemini"
    print("✅ Provider GEMINI aktif!")
else:
    print("ℹ️ Kunci API dilewati. Anda tetap bisa mencoba Modul 00 (Tanpa API Key) dan Modul 04 (Audio Gratis)!")

--- 
## 🆓 Modul 00: LLM 100% Gratis Tanpa API Key Sama Sekali

Model open-source (`Qwen/Qwen2.5-0.5B-Instruct`) berjalan langsung di GPU/CPU Google Colab tanpa membutuhkan pendaftaran, akun, maupun API Key.

In [ ]:
from transformers import pipeline
import torch

print("Sedang memuat model open-source ke memori Colab...")
device = 0 if torch.cuda.is_available() else -1

free_llm = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype=torch.float16 if device == 0 else torch.float32,
    device=device
)

prompt_uji = "Jelaskan dalam 2 kalimat apa itu Cloud Computing dan apa keuntungannya bagi UMKM?"
messages = [
    {"role": "system", "content": "Anda adalah asisten AI yang ramah dalam bahasa Indonesia."},
    {"role": "user", "content": prompt_uji}
]

output = free_llm(messages, max_new_tokens=150, temperature=0.3)
print("\n--- Respon Model Bebas Biaya (Tanpa API Key) ---")
print(output[0]["generated_text"][-1]["content"].strip())

--- 
## 💬 Modul 01: Chat Client Interaktif (Multi-turn Conversation)

Menggunakan API OpenAI-compatible yang terhubung ke model Gemini / Azure untuk percakapan berbasis state.

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY", "")
)

chat_history = [
    {"role": "system", "content": "Anda adalah staf customer service toko laptop. Jawab singkat dan ramah."},
    {"role": "user", "content": "Halo, laptop apa yang cocok untuk mahasiswa teknik sipil?"}
]

response = client.chat.completions.create(
    model="gemini-1.5-flash",
    messages=chat_history,
    temperature=0.3
)

jawaban = response.choices[0].message.content
print("Asisten:", jawaban)

--- 
## 🤖 Modul 02: Autonomous AI Agent dengan Tool Calling (Function Calling)

Model memutuskan sendiri kapan harus memanggil fungsi lokal `cek_ongkir` dan `cek_resi`.

In [ ]:
# Jalankan skrip agen alternatif yang telah disiapkan
!python alternatif/02_agent_alt.py

--- 
## 📊 Modul 03: Analisis Teks Berstruktur (JSON Mode & NER)

Membaca dataset `data/ulasan.txt` dan mengekstrak entitas bernama, sentimen, dan ringkasan ke dalam skema JSON tervalidasi.

In [ ]:
# Jalankan modul analisis teks JSON mode
!python alternatif/03_text_analysis_alt.py

# Tampilkan 1 sampel hasil ekstraksi JSON
import json
with open("output/hasil_analisis_alt.json", encoding="utf-8") as f:
    hasil = json.load(f)
print("Sampel JSON Hasil Analisis:")
print(json.dumps(hasil[0], ensure_ascii=False, indent=2))

--- 
## 🎙️ Modul 04: Layanan Suara Gratis (Edge-TTS) & Pemutar Audio Colab

Sintesis suara natural bahasa Indonesia (`id-ID-ArdiNeural`) secara **100% gratis tanpa API Key**!

In [ ]:
import edge_tts
import asyncio
from IPython.display import Audio, display

teks = "Halo! Selamat datang di modul praktik AI 901. Ini adalah suara sintesis neural bahasa Indonesia gratis."
voice = "id-ID-ArdiNeural"
audio_path = "output/pengumuman_colab.mp3"
os.makedirs("output", exist_ok=True)

async def generate_audio():
    comm = edge_tts.Communicate(teks, voice)
    await comm.save(audio_path)

await generate_audio()
print("✅ Audio berhasil disintesis!")
display(Audio(audio_path, autoplay=True))

--- 
## 👁️ Modul 05: Multimodal Computer Vision (Foto Rak Toko)

Menganalisis gambar fisik `data/rak.jpg` untuk mendeteksi dan menghitung produk di rak display.

In [ ]:
from IPython.display import Image, display

# Tampilkan foto rak yang diuji
display(Image("data/rak.jpg", width=400))

# Jalankan skrip analisis vision multimodal
!python alternatif/05_vision_alt.py

--- 
## 📑 Modul 06: Document Understanding & Ekstraksi Faktur Invoice PDF

Membaca file PDF faktur `data/invoice.pdf` dan mengekstrak field terstruktur (Vendor, Total, Item, Tanggal) ke JSON.

In [ ]:
# Jalankan ekstraksi dokumen invoice
!python alternatif/06_documents_alt.py

--- 
## 🎓 Tips & Kisi-Kisi Ujian Sertifikasi AI-901 / AI-900

1. **`temperature = 0.0` vs `1.0`:** Nilai rendah dipakai untuk faktual/finansial/analisis data; nilai tinggi untuk ide kreatif/marketing.
2. **Pola Asynchronous REST:** Permintaan analisis dokumen mengembalikan `HTTP 202 Accepted`, dan URL polling berada di header **`Operation-Location`**.
3. **Tool Calling:** Model AI **tidak** mengeksekusi kode Python sendiri; model mengembalikan argumen fungsi dalam format JSON, lalu aplikasi lokal mengeksekusinya.
4. **SSML (Speech Synthesis Markup Language):** Standar XML untuk mengatur jeda (`<break time="..."/>`), intonasi (`<prosody>`), dan karakter suara (`<voice>`).